# YOLO26 学習用 Colab ノートブック

`shared/train.py` を Google Colab で実行するための notebook です。

前提:
- GPU ランタイムを有効化して使う
- データセットは `train/`, `valid/`, `test/` を含む YOLO 形式で Google Drive に配置
- 必要なら事前学習済み重みを Google Drive に置いておく

ポイント:
- リポジトリは `koshien2015/ultralytics` の `feature/small-object-detection-pipeline` ブランチを使う
  （argparse 版 train.py はこのブランチにのみ存在する）
- 学習 run の出力先は `--project` で **Drive 直下に書き込む**。Colab が途中で切断されても
  チェックポイントが Drive に残り、`resume=True` での完全再開が新セッションでも可能になる
- `WEIGHTS_PATH` を指定すると、既存の `.pt` を初期重みにした追加学習ができます
- 中断した run の完全再開（optimizer state 込み）は下部の「完全再開」セルを使います


In [ ]:
# Colab で GPU が見えているか確認
!nvidia-smi


In [ ]:
# 学習データ・重み・run出力の置き場として Google Drive を使う（必須）
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import textwrap

# 必要に応じて変更してください
REPO_URL = "https://github.com/koshien2015/ultralytics.git"
REPO_BRANCH = "feature/small-object-detection-pipeline"
REPO_DIR = Path("/content/ultralytics")
ULTRALYTICS_DIR = REPO_DIR  # リポジトリ自体が ultralytics フォーク
SHARED_DIR = REPO_DIR / "shared"

# Drive 上の学習データ配置例:
# /content/drive/MyDrive/cap-baseball/yolo-dataset/
#   ├─ train/
#   ├─ valid/
#   └─ test/
DATASET_DIR = Path("/content/drive/MyDrive/cap-baseball/yolo-dataset")

# 追加学習したい既存モデル。不要なら None。
WEIGHTS_PATH = Path("/content/drive/MyDrive/cap-baseball/weights/yolo26m.pt")
# WEIGHTS_PATH = None

MODEL_CFG = "yolo26m-p2.yaml"
IMGSZ = 1280
EPOCHS = 20
BATCH = -1  # VRAM に合わせて自動調整（固定したい場合のみ数値を指定）
RUN_NAME = "cap_yolo26_colab"

# run の出力先（Drive 直書き）。切断対策と resume のため Drive 上を指定する
PROJECT_DIR = Path("/content/drive/MyDrive/cap-baseball/training-results")

# 完全再開したい場合はこちら（PROJECT_DIR 配下の last.pt を指す）
RESUME_TRAINING = False
RESUME_WEIGHTS_PATH = PROJECT_DIR / RUN_NAME / "weights" / "last.pt"

print(f"DATASET_DIR: {DATASET_DIR}")
print(f"WEIGHTS_PATH: {WEIGHTS_PATH}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"RESUME_TRAINING: {RESUME_TRAINING}")
print(f"RESUME_WEIGHTS_PATH: {RESUME_WEIGHTS_PATH}")


## モードの使い分け

- 新規学習: `RESUME_TRAINING = False` かつ `WEIGHTS_PATH = None`
- 追加学習: `RESUME_TRAINING = False` かつ `WEIGHTS_PATH` を指定
- 完全再開: `RESUME_TRAINING = True` にして、**学習セルではなく下部の「完全再開」セルを実行する**
  （train.py 経由の再開はアーキテクチャ不一致の恐れがあるため学習セルはスキップされる）


In [ ]:
if REPO_DIR.exists():
    print(f"Repo already exists: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], check=True)
# ultralytics 本体を editable install（torch / opencv などの依存もここで入る）
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ULTRALYTICS_DIR)], check=True)

print("Setup complete")


In [ ]:
required_dirs = [DATASET_DIR / "train", DATASET_DIR / "valid"]
missing = [str(path) for path in required_dirs if not path.exists()]
if missing:
    raise FileNotFoundError("Missing dataset directories: " + ", ".join(missing))

if RESUME_TRAINING:
    if not RESUME_WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"Resume weights file not found: {RESUME_WEIGHTS_PATH}")
else:
    if WEIGHTS_PATH is not None and not WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"Weights file not found: {WEIGHTS_PATH}")

data_yaml_path = SHARED_DIR / "data-colab.yaml"
data_yaml_path.write_text(
    textwrap.dedent(
        f"""
        path: {DATASET_DIR.as_posix()}
        train: train
        val: valid
        test: test

        nc: 11
        names:
          0: cap
          1: pitcher_motion
          2: batter_stance
          3: umpire
          4: catcher
          5: pitcher_release
          6: batter_swing
          7: catcher_stance
          8: catcher_catch
          9: catcher_throw
          10: catcher_miss
        """
    ).strip() + "\n",
    encoding="utf-8",
)

print(data_yaml_path.read_text(encoding="utf-8"))


In [ ]:
# 学習の実行（新規学習・追加学習用。完全再開は下のセルを使う）
os.chdir(SHARED_DIR)

if RESUME_TRAINING:
    print("RESUME_TRAINING = True のため、このセルはスキップします。")
    print("下部の「完全再開」セルを実行してください。")
else:
    command = [
        sys.executable, "train.py",
        "--data", "data-colab.yaml",
        "--model", MODEL_CFG,
        "--imgsz", str(IMGSZ),
        "--epochs", str(EPOCHS),
        "--batch", str(BATCH),
        "--name", RUN_NAME,
        "--project", str(PROJECT_DIR),
    ]
    if WEIGHTS_PATH is not None:
        command.extend(["--weights", str(WEIGHTS_PATH)])

    print(" ".join(command))
    subprocess.run(command, check=True)


## 完全再開（optimizer state 込みの厳密な再開）

run 出力は `--project` で Drive に直書きしているため、Colab のセッションが変わっても
`last.pt` とその run ディレクトリは有効です。`RESUME_TRAINING = True` にして
このセルを実行すると、中断したエポックから学習を再開します。


In [ ]:
# RESUME_TRAINING = True のときだけ実行
from ultralytics import YOLO

if RESUME_TRAINING:
    model = YOLO(str(RESUME_WEIGHTS_PATH))
    model.train(resume=True)
else:
    print("RESUME_TRAINING が False なのでこのセルは不要です")


In [ ]:
# run 出力の確認（--project により Drive に直接書き込まれている）
run_dir = PROJECT_DIR / RUN_NAME
print(f"run_dir: {run_dir}")
print(f"exists: {run_dir.exists()}")

if run_dir.exists():
    for p in sorted(run_dir.iterdir()):
        print(" ", p.name)


## 学習済みモデルのダウンロード

run 出力は Drive に保存済みなのでダウンロードは必須ではありませんが、
手元にすぐ落としたい場合は以下を実行してください。


In [ ]:
weights_dir = run_dir / "weights"
best_pt = weights_dir / "best.pt"
last_pt = weights_dir / "last.pt"

print(f"best.pt: {best_pt} exists={best_pt.exists()}")
print(f"last.pt: {last_pt} exists={last_pt.exists()}")


In [ ]:
from google.colab import files

DOWNLOAD_BEST = True
DOWNLOAD_LAST = False

if DOWNLOAD_BEST:
    if not best_pt.exists():
        raise FileNotFoundError(f"best.pt not found: {best_pt}")
    files.download(str(best_pt))

if DOWNLOAD_LAST:
    if not last_pt.exists():
        raise FileNotFoundError(f"last.pt not found: {last_pt}")
    files.download(str(last_pt))
